# ReadyNow! — FEMA Emergency Preparedness Agent

Case study: multi-agent ADK system providing weather alerts, disaster news,
evacuation routes, and safety guidance. See architecture diagram in repo
README.

## Setup

In [1]:
import vertexai
from google.adk.agents import Agent, SequentialAgent
from google.adk.tools import google_search
from dotenv import load_dotenv
import os
from datetime import date

load_dotenv()

PROJECT_ID = "adroit-sol-501711-r0"
LOCATION = "us-central1"
STAGING_BUCKET = "gs://adroit-sol-501711-r0-adk-staging"

vertexai.init(
    project=PROJECT_ID,
    location=LOCATION,
    staging_bucket=STAGING_BUCKET,
)

TODAY = date.today().strftime("%B %d, %Y")

print("Vertex AI initialized. Today is:", TODAY)

Vertex AI initialized. Today is: July 10, 2026


/usr/local/lib/python3.12/dist-packages/google/adk/features/_feature_decorator.py:72: UserWarning: [EXPERIMENTAL] feature FeatureName.PLUGGABLE_AUTH is enabled.
  check_feature_enabled()


In [11]:
import os

os.environ["GOOGLE_GEO_API_KEY"] = "YOUR_GOOGLE_GEO_API_KEY_HERE"

print("Environment variable set.")

Environment variable set.


## Tools and callbacks

Written to disk via `%%writefile` so they can be bundled at deployment
time (Agent Engine doesn't auto-package local notebook-defined objects —
only files listed in `extra_packages`).

In [12]:
%%writefile tools.py
import os
import requests


def get_current_location() -> dict:
    """
    Detects the user's current location using IP-based geolocation
    via the Google Geolocation API.

    Returns:
        dict: A dictionary with 'status', 'lat', and 'lon' keys.
    """
    google_key = os.getenv("GOOGLE_GEO_API_KEY")
    geo_url = f"https://www.googleapis.com/geolocation/v1/geolocate?key={google_key}"
    try:
        response = requests.post(geo_url, json={})
        response.raise_for_status()
        data = response.json()
        return {
            "status": "success",
            "lat": data["location"]["lat"],
            "lon": data["location"]["lng"],
        }
    except Exception as e:
        return {"status": "error", "error_message": str(e)}


def get_coordinates_for_city(city: str) -> dict:
    """
    Looks up latitude and longitude for a city name using the Google
    Geocoding API.

    Args:
        city: Name of the city, e.g. "Kansas City" or "Topeka, KS".

    Returns:
        dict: A dictionary with 'status', 'lat', 'lon', and 'resolved_name'.
    """
    google_key = os.getenv("GOOGLE_GEO_API_KEY")
    try:
        geo_url = "https://maps.googleapis.com/maps/api/geocode/json"
        params = {"address": city, "key": google_key}
        response = requests.get(geo_url, params=params)
        response.raise_for_status()
        data = response.json()
        if data.get("status") != "OK" or not data.get("results"):
            return {
                "status": "error",
                "error_message": f"No location found for '{city}' (status: {data.get('status')})",
            }
        result = data["results"][0]
        location = result["geometry"]["location"]
        return {
            "status": "success",
            "lat": location["lat"],
            "lon": location["lng"],
            "resolved_name": result["formatted_address"],
        }
    except Exception as e:
        return {"status": "error", "error_message": str(e)}


def get_weather(lat: float, lon: float) -> dict:
    """
    Fetches current weather conditions and active alerts from the
    National Weather Service for given coordinates.

    Args:
        lat: Latitude of the location.
        lon: Longitude of the location.

    Returns:
        dict: A dictionary with 'status', current conditions, and any
        active weather alerts for the area.
    """
    headers = {"User-Agent": "readynow-app (emergency-preparedness@example.com)"}
    try:
        points_url = f"https://api.weather.gov/points/{lat},{lon}"
        points_resp = requests.get(points_url, headers=headers)
        points_resp.raise_for_status()
        points_data = points_resp.json()["properties"]
        forecast_url = points_data["forecast"]

        forecast_resp = requests.get(forecast_url, headers=headers)
        forecast_resp.raise_for_status()
        current = forecast_resp.json()["properties"]["periods"][0]

        zone_url = points_data.get("forecastZone", "")
        alerts_url = f"https://api.weather.gov/alerts/active?zone={zone_url.split('/')[-1]}"
        alerts_resp = requests.get(alerts_url, headers=headers)
        alerts_data = alerts_resp.json() if alerts_resp.ok else {"features": []}
        active_alerts = [
            {
                "event": f["properties"]["event"],
                "headline": f["properties"]["headline"],
                "severity": f["properties"]["severity"],
            }
            for f in alerts_data.get("features", [])
        ]

        return {
            "status": "success",
            "name": current["name"],
            "temperature": current["temperature"],
            "unit": current["temperatureUnit"],
            "forecast": current["shortForecast"],
            "active_alerts": active_alerts,
        }
    except Exception as e:
        return {"status": "error", "error_message": str(e)}


def get_safe_route(origin: str, destination: str) -> dict:
    """
    Computes driving directions from an origin to a destination using
    the Google Routes API, intended for evacuation route guidance.

    Args:
        origin: Starting address or place name.
        destination: Destination address or place name (e.g. a shelter
        or safe zone).

    Returns:
        dict: A dictionary with 'status', total distance, duration, and
        a list of step-by-step navigation instructions.
    """
    google_key = os.getenv("GOOGLE_GEO_API_KEY")
    url = "https://routes.googleapis.com/directions/v2:computeRoutes"
    headers = {
        "Content-Type": "application/json",
        "X-Goog-Api-Key": google_key,
        "X-Goog-FieldMask": "routes.duration,routes.distanceMeters,routes.legs.steps.navigationInstruction",
    }
    payload = {
        "origin": {"address": origin},
        "destination": {"address": destination},
        "travelMode": "DRIVE",
    }
    try:
        response = requests.post(url, json=payload, headers=headers)
        response.raise_for_status()
        data = response.json()
        if not data.get("routes"):
            return {"status": "error", "error_message": "No route found."}

        route = data["routes"][0]
        steps = [
            step["navigationInstruction"]["instructions"]
            for leg in route["legs"]
            for step in leg["steps"]
            if "navigationInstruction" in step
        ]
        return {
            "status": "success",
            "distance_meters": route["distanceMeters"],
            "duration": route["duration"],
            "steps": steps,
        }
    except Exception as e:
        return {"status": "error", "error_message": str(e)}

Overwriting tools.py


In [13]:
%%writefile callbacks.py
import logging
import re
from typing import Optional

from google.adk.agents.callback_context import CallbackContext
from google.adk.models import LlmRequest, LlmResponse
from google.genai import types

logger = logging.getLogger("readynow_callbacks")
logger.setLevel(logging.INFO)
logger.propagate = False

if not logger.handlers:
    file_handler = logging.FileHandler("readynow.log")
    file_handler.setFormatter(
        logging.Formatter("%(asctime)s [%(levelname)s] %(message)s")
    )
    logger.addHandler(file_handler)


def _extract_user_text(llm_request: LlmRequest) -> str:
    try:
        last_content = llm_request.contents[-1]
        return "".join(part.text or "" for part in last_content.parts)
    except (IndexError, AttributeError):
        return "<unavailable>"


def log_before_model(callback_context: CallbackContext, llm_request: LlmRequest) -> Optional[LlmResponse]:
    """Logs the user-provided request before it's sent to the LLM."""
    user_text = _extract_user_text(llm_request)
    logger.info(
        f"REQUEST | agent={callback_context.agent_name} | "
        f"invocation={callback_context.invocation_id} | user_input={user_text!r}"
    )
    return None


def validate_input(callback_context: CallbackContext, llm_request: LlmRequest) -> Optional[LlmResponse]:
    """
    Blocks requests containing disallowed content (hard filter).
    Mission-relevance refusal is handled separately by the root agent's
    own instructions, since that judgment is semantic rather than a
    fixed keyword match.
    """
    user_text = _extract_user_text(llm_request)
    if re.search(r"\bBAD\b", user_text.upper()):
        logger.info(
            f"BLOCKED | agent={callback_context.agent_name} | "
            f"invocation={callback_context.invocation_id} | reason=content_guideline_violation"
        )
        return LlmResponse(
            content=types.Content(
                role="model",
                parts=[types.Part(text="Message violates our content guidelines")],
            )
        )
    return None


def log_after_model(callback_context: CallbackContext, llm_response: LlmResponse) -> Optional[LlmResponse]:
    """Logs the model's response after it's received."""
    try:
        response_text = "".join(part.text or "" for part in llm_response.content.parts)
    except AttributeError:
        response_text = "<unavailable>"
    logger.info(
        f"RESPONSE | agent={callback_context.agent_name} | "
        f"invocation={callback_context.invocation_id} | response={response_text!r}"
    )
    return None

Overwriting callbacks.py


In [14]:
from tools import get_current_location, get_coordinates_for_city, get_weather, get_safe_route
from callbacks import log_before_model, validate_input, log_after_model

print("Tools and callbacks imported.")

Tools and callbacks imported.


## Agents

Four specialist teams (search → critique → refine each), plus a root
agent that describes its capabilities, enforces mission-scope refusal,
and delegates to the right specialist.

**Note:** `search_critique`/`search_refine` are told today's actual date
explicitly. An earlier version relied on a generic "trust the search
tool" instruction, but the underlying model's training data predates
July 2026, so it kept "correcting" real, correctly-dated search results —
once flagging accurate wildfire data as fabricated, and once silently
rewriting a real date to a wrong one. Stating today's date directly as
verified ground truth resolved this.

In [15]:
GEMINI_MODEL = "gemini-2.5-flash"


# --- Weather team ---

weather_specialist = Agent(
    name="weather_specialist",
    model=GEMINI_MODEL,
    description="Provides current weather conditions and active alerts for a location.",
    instruction=(
        "You help users get real-time weather information and active alerts "
        "during emergencies. If the user gives a city name, call "
        "get_coordinates_for_city to find its coordinates, then call "
        "get_weather. If they don't specify a location, call "
        "get_current_location first. Always mention any active alerts "
        "clearly and prominently, since these may be safety-critical."
    ),
    tools=[get_current_location, get_coordinates_for_city, get_weather],
    output_key="draft_response",
)

weather_critique = Agent(
    name="weather_critique",
    model=GEMINI_MODEL,
    description="Reviews weather/alert responses for clarity and completeness.",
    instruction=(
        "Review this weather response: {draft_response}\n\n"
        "Check that active alerts (if any) are clearly highlighted, units "
        "are unambiguous, and the language is simple enough for someone in "
        "a stressful situation to understand quickly. Provide specific "
        "suggestions. Do not rewrite it yourself."
    ),
    output_key="critique",
)

weather_refine = Agent(
    name="weather_refine",
    model=GEMINI_MODEL,
    description="Rewrites the weather response based on critique.",
    instruction=(
        "Original response: {draft_response}\n"
        "Critique: {critique}\n\n"
        "Rewrite the response applying the critique's suggestions. Keep it "
        "clear, calm, and easy to scan quickly. Output ONLY the final "
        "response text."
    ),
)

weather_team = SequentialAgent(
    name="weather_team",
    description="Answers weather and alert questions, then validates and refines the response.",
    sub_agents=[weather_specialist, weather_critique, weather_refine],
)


# --- Search team ---

search_specialist = Agent(
    name="search_specialist",
    model=GEMINI_MODEL,
    description="Searches the web for current news and disaster-related information.",
    instruction=(
        "Search the web for current, relevant information to answer the "
        "user's question about a disaster, emergency, or news event. "
        "Prioritize official sources (government agencies, news outlets) "
        "over unverified content."
    ),
    tools=[google_search],
    disallow_transfer_to_parent=True,
    disallow_transfer_to_peers=True,
    output_key="draft_response",
)

search_critique = Agent(
    name="search_critique",
    model=GEMINI_MODEL,
    description="Reviews search-based responses for accuracy and clarity.",
    instruction=(
        f"Today's actual date is {TODAY}. Search results reflecting this "
        f"date or nearby dates are current and correct, not future or "
        f"fabricated. Do not flag or change any date that matches or "
        f"precedes today's date — this is verified ground truth, not an "
        f"error.\n\n"
        "Review this response: {draft_response}\n\n"
        "Check that it's accurate, cites credible information, and is easy "
        "to understand under stress. Provide specific suggestions. Do not "
        "rewrite it yourself."
    ),
    output_key="critique",
)

search_refine = Agent(
    name="search_refine",
    model=GEMINI_MODEL,
    description="Rewrites the search response based on critique.",
    instruction=(
        f"Today's actual date is {TODAY}. Never change, remove, or "
        f"'correct' a date in the original response that matches or "
        f"precedes today's date — those dates are already correct.\n\n"
        "Original response: {draft_response}\n"
        "Critique: {critique}\n\n"
        "Rewrite the response applying the critique's suggestions, except "
        "for any suggestion to change a correct date. Output ONLY the "
        "final response text."
    ),
)

search_team = SequentialAgent(
    name="search_team",
    description="Answers news/search questions, then validates and refines the response.",
    sub_agents=[search_specialist, search_critique, search_refine],
)


# --- Routes team ---

routes_specialist = Agent(
    name="routes_specialist",
    model=GEMINI_MODEL,
    description="Suggests evacuation or safety routes between two locations.",
    instruction=(
        "Help the user find a route to safety. Ask for or infer an origin "
        "and destination (e.g. their current location and a named shelter, "
        "city, or safe zone), then call get_safe_route. Present the route "
        "as clear, numbered steps with total distance and estimated time."
    ),
    tools=[get_safe_route],
    output_key="draft_response",
)

routes_critique = Agent(
    name="routes_critique",
    model=GEMINI_MODEL,
    description="Reviews route responses for clarity and safety framing.",
    instruction=(
        "Review this route response: {draft_response}\n\n"
        "Check that the steps are easy to follow quickly, distance/time are "
        "clearly stated, and nothing is ambiguous. Provide specific "
        "suggestions. Do not rewrite it yourself."
    ),
    output_key="critique",
)

routes_refine = Agent(
    name="routes_refine",
    model=GEMINI_MODEL,
    description="Rewrites the route response based on critique.",
    instruction=(
        "Original response: {draft_response}\n"
        "Critique: {critique}\n\n"
        "Rewrite the response applying the critique's suggestions. Output "
        "ONLY the final response text."
    ),
)

routes_team = SequentialAgent(
    name="routes_team",
    description="Suggests evacuation routes, then validates and refines the response.",
    sub_agents=[routes_specialist, routes_critique, routes_refine],
)


# --- Q&A team ---

qa_specialist = Agent(
    name="qa_specialist",
    model=GEMINI_MODEL,
    description="Answers general emergency preparedness and safety questions.",
    instruction=(
        "Answer general questions about emergency preparedness, disaster "
        "safety, and what to do before/during/after an emergency, using "
        "your own knowledge. Be practical and specific."
    ),
    output_key="draft_response",
)

qa_critique = Agent(
    name="qa_critique",
    model=GEMINI_MODEL,
    description="Reviews general safety answers for accuracy and clarity.",
    instruction=(
        "Review this answer: {draft_response}\n\n"
        "Check it's accurate, practical, and easy to understand quickly. "
        "Provide specific suggestions. Do not rewrite it yourself."
    ),
    output_key="critique",
)

qa_refine = Agent(
    name="qa_refine",
    model=GEMINI_MODEL,
    description="Rewrites the Q&A response based on critique.",
    instruction=(
        "Original response: {draft_response}\n"
        "Critique: {critique}\n\n"
        "Rewrite the response applying the critique's suggestions. Output "
        "ONLY the final response text."
    ),
)

qa_team = SequentialAgent(
    name="qa_team",
    description="Answers general safety questions, then validates and refines the response.",
    sub_agents=[qa_specialist, qa_critique, qa_refine],
)

print("Four specialist teams defined.")

Four specialist teams defined.


In [16]:
root_agent = Agent(
    name="root_agent",
    model=GEMINI_MODEL,
    description=(
        "ReadyNow! is an emergency preparedness assistant. It provides "
        "real-time weather alerts, disaster news, evacuation routes, and "
        "safety guidance based on your location and situation."
    ),
    instruction=(
        "You are ReadyNow!, an emergency preparedness assistant built for FEMA. "
        "If the user asks what you can do, describe your capabilities clearly: "
        "real-time weather and alerts, current disaster news, suggested "
        "evacuation routes, and general safety guidance.\n\n"
        "For every other request, first judge whether it relates to your "
        "mission: emergencies, disasters, weather, safety, or evacuation. "
        "If it is unrelated (e.g. general trivia, coding help, entertainment), "
        "politely decline and explain that you can only help with emergency "
        "preparedness and safety topics.\n\n"
        "If the request is on-mission, delegate to exactly one team:\n"
        "- weather_team for weather conditions or alerts\n"
        "- search_team for current news or disaster information\n"
        "- routes_team for evacuation routes or directions to safety\n"
        "- qa_team for general safety/preparedness questions\n"
    ),
    sub_agents=[weather_team, search_team, routes_team, qa_team],
    before_model_callback=[log_before_model, validate_input],
    after_model_callback=log_after_model,
)

print("root_agent assembled:", root_agent.name)

root_agent assembled: root_agent


## Local testing

Exercises all five capability paths via `AdkApp` before deployment.

In [17]:
from vertexai.preview.reasoning_engines import AdkApp

app = AdkApp(agent=root_agent)

print("AdkApp created.")

AdkApp created.


In [18]:
# Test 1: Capability description
for event in app.stream_query(
    user_id="test_user",
    message="What can you do?",
):
    print(event)

{'model_version': 'gemini-2.5-flash', 'content': {'parts': [{'text': 'As ReadyNow!, an emergency preparedness assistant, I can help you with a variety of emergency-related tasks. I can provide:\n\n*   **Real-time weather information and alerts:** I can give you the latest weather updates and inform you about any severe weather alerts in your area.\n*   **Current disaster news:** I can provide information on ongoing disasters and emergencies.\n*   **Suggested evacuation routes:** If needed, I can help you find safe evacuation routes.\n*   **General safety guidance:** I can answer your questions about emergency preparedness and general safety tips.', 'thought_signature': 'CsICAY89a1_ohFqqB9GKLpql5Vk_TkYq52FdzxuY_Ypvi7nNwQJ5B0_x9zXmkZ6hBOaYRfSZn0RZKp_hEm5j7M2bD8q6kT9yaxGjwtYURXoE1eM_a59dx1yyih3GA85PCXAdmtxLxUG3i8zfuI8rq8dkK1HWEpaF4FXjjA6nvx3bv3u0LrYpvYVU5yIyTxXH1LD9igKLBU0FC-lngkMHpCE_6h-0-oRSz_MrQ8_LlXY4GcP3lpTW1zXeWh7ZxeEzI6uX6_qLtk9hLjZdjG7E_vKg-fjV2KRzXNAxdD2-uFFgAOTmAbLRUxSM3ceybUyqx

In [19]:
# Test 2: Mission-scope refusal
for event in app.stream_query(
    user_id="test_user",
    message="Can you help me write a Python script to sort a list?",
):
    print(event)

{'model_version': 'gemini-2.5-flash', 'content': {'parts': [{'text': 'I can only help with emergency preparedness and safety topics. I cannot help with writing Python scripts.', 'thought_signature': 'CqQCAY89a1_6JiX200_zMV-Ir-sTdXFy1JM3nZjQvcXia_t33DzFmR_TrcVBSduLCcs_ittr6In2Khuju7_ejCboT6CoarLtm33XUQm8qkUBgTvKBO-cAwbl6B8LmiP119RtWOQexPakYkjmxfbuAhwK6DDZGmCY7Q7g8grZsjhkSOF5Pq-4CsIz6ix6C6jc80uggl-584_OHOu83AR1LFOxL4EWNuCfBNKAgw_P4bEmwRZW4UyjeolRy-v88zldu3iStW6EmrEL548tBtBJd-y3d-PJqVqQ4XxJH_j5vJvySyl0D8pC7V4uTG14L8yCohIUnBPKozf7M1CAumLjKOYZ4Y1hafyTRvmey4QPFFgY2rIkTNqd6EDprCwVF_vOfJI4r1ZQnw=='}], 'role': 'model'}, 'finish_reason': 'STOP', 'usage_metadata': {'candidates_token_count': 19, 'candidates_tokens_details': [{'modality': 'TEXT', 'token_count': 19}], 'prompt_token_count': 585, 'prompt_tokens_details': [{'modality': 'TEXT', 'token_count': 585}], 'thoughts_token_count': 51, 'total_token_count': 655, 'traffic_type': 'ON_DEMAND'}, 'avg_logprobs': -0.5921701130114103, 'invocation_id': '

In [20]:
# Test 3: Weather/alerts
for event in app.stream_query(
    user_id="test_user",
    message="What's the weather like in Kansas City, MO right now, any alerts?",
):
    print(event)

{'model_version': 'gemini-2.5-flash', 'content': {'parts': [{'function_call': {'id': 'adk-dc8643a2-eb4e-4d64-8b53-d7fc3a9f7232', 'args': {'agent_name': 'weather_team'}, 'name': 'transfer_to_agent'}, 'thought_signature': 'Cs4CAY89a19iSwEFCEZa6UzyIbiVUHosnwlGZ7PPyncbqncIWmUJU4QhzEMo6busZYujWfNjUF9_7ERJ-mzBxU0XuXt-7AUpxowUqPFz4UmeUKSJOkfoIUj7yYuO46Sz7KhxIU2EXEc_Lcojb2adPZ7DMB8kxHLqRU_BMH6rdfG4_V9fiLhdl_MvJoJLL1dNF7p3OZwLAVDRSRxTYSfyljLCeqct-CKU1bjUpGEQLIJJag8umaL7KIIWs943hrZP82Sf0kfb1QKQw5udgXKc5OVC95gMTW3PSWHSvIAer-ZodicPxNX8yHz3k1iZLBVrrr3VPVmakkD85Hvfr2q4NV0lhWxszCtiwZiKApgr3YQrHHBlNcaQpvIV4sOyRh9b-ViTF98E0aa84DnGNGqkDjdCUxhcRazL0HvZCXu3DPtoZZHX1OHdptvRZXAq-FkiDw=='}], 'role': 'model'}, 'finish_reason': 'STOP', 'usage_metadata': {'candidates_token_count': 11, 'candidates_tokens_details': [{'modality': 'TEXT', 'token_count': 11}], 'prompt_token_count': 589, 'prompt_tokens_details': [{'modality': 'TEXT', 'token_count': 589}], 'thoughts_token_count': 62, 'total_token_count': 662, 'traffic

In [21]:
# Test 4: Routes
for event in app.stream_query(
    user_id="test_user",
    message="I need a safe route from Kansas City, MO to Topeka, KS",
):
    print(event)

{'model_version': 'gemini-2.5-flash', 'content': {'parts': [{'function_call': {'id': 'adk-970f91e7-311b-45f4-bda1-61de17e7b93b', 'args': {'agent_name': 'routes_team'}, 'name': 'transfer_to_agent'}, 'thought_signature': 'CqgCAY89a19oy6uzIOl8BDJZENhaFdZF8gNKtrHndUz1gGWdR7lbX8q1xpu9XgPR4FlxBwAw_ewIV6OX_-bL_UEznbR-nVHsYBQ6xMAHoe4CpaXhUlrl604zKLmaKmV6jk4sdu8GrdUMk3Ewhv-kkqWjlKzC1zD8xRcQQX7X8kLy8tQJT7yby_U3JSlpbqUAI5G-33a2Dnpc2Za_BfXn3oqwuoMYjWxgyeNZ1ROytnGtoZoqTAcodMBC-KVbZ4LOV_WY43Px13UvTctv0HXPNsi9AOpl2li1km98i3AcVBSaDyeglyt1CfpOINbrJ-En64oQf-VGon5UlB4Uh-LbhOIn1W0bR2PDbOGXyAFtaryxDSQcLhZ3AjE12kGq9RWROtCrpce5qmnGKYM='}], 'role': 'model'}, 'finish_reason': 'STOP', 'usage_metadata': {'candidates_token_count': 11, 'candidates_tokens_details': [{'modality': 'TEXT', 'token_count': 11}], 'prompt_token_count': 586, 'prompt_tokens_details': [{'modality': 'TEXT', 'token_count': 586}], 'thoughts_token_count': 56, 'total_token_count': 653, 'traffic_type': 'ON_DEMAND'}, 'avg_logprobs': -0.519444335590

In [22]:
# Test 5: Search/news
for event in app.stream_query(
    user_id="test_user",
    message="Are there any major wildfires happening right now?",
):
    print(event)

{'model_version': 'gemini-2.5-flash', 'content': {'parts': [{'function_call': {'id': 'adk-899a144e-6792-4b03-97a7-c5e0022e1f98', 'args': {'agent_name': 'search_team'}, 'name': 'transfer_to_agent'}, 'thought_signature': 'CrUCAY89a18aZFs4uybYV4oAD5JFj58-jszPGpHflMXqnPuvVwzeYU39ystDx3YGL4JdUy-iri7de8Plqetu2mYJkaQJDEhla3BFFIzDzN0oSJeFw7qXz3L07ApPy4e2baTRi_nSLgs3VzQPwkOIotxvnpuXZM_7lIKyEsUgaoXekMNDd8De9sMMwhiZGxOXacPlhGbaZ_L-MXH5-10MwRIqs_5uvox7Lh5NHYNxy5hNIla8iWbQ-BXWK2EP889lOvRKcLy_9G5fEqXKxZst-M-NWcbqog4JdHxSfqgau2ISOiWYd3tyW8M-d3UMaIdeBxxZK_Z75d9hiNA2aCSg-k0Fct_VIE7X1ClIK_iVkYwquwZ9EleaRKGPCXk-MsluNnG1Iv184GYD9lWL-oI1gwfZoY15j8NY'}], 'role': 'model'}, 'finish_reason': 'STOP', 'usage_metadata': {'candidates_token_count': 11, 'candidates_tokens_details': [{'modality': 'TEXT', 'token_count': 11}], 'prompt_token_count': 581, 'prompt_tokens_details': [{'modality': 'TEXT', 'token_count': 581}], 'thoughts_token_count': 55, 'total_token_count': 647, 'traffic_type': 'ON_DEMAND'}, 'avg_logprobs':

## Deployment to Agent Platform (Vertex AI Agent Engine)

**Note:** `google-adk` must be pinned to the exact locally-installed
version. An earlier deployment without this pin resolved a newer ADK
release at deploy time, whose `Runner` expected an `agent.mode` attribute
that didn't exist on the older `LlmAgent` class used to pickle this
agent — causing `stream_query` to fail silently (0 events, no exception
in the client) with the real error only visible in Cloud Logging
(`resource.type="aiplatform.googleapis.com/ReasoningEngine"`).

In [23]:
import google.adk
print("Local ADK version:", google.adk.__version__)

Local ADK version: 1.29.0


In [24]:
from vertexai import agent_engines

remote_agent = agent_engines.create(
    app,
    requirements=[
        "google-cloud-aiplatform[agent_engines,adk]",
        "google-adk==1.29.0",  # pinned to match local version above
    ],
    extra_packages=["tools.py", "callbacks.py"],
    env_vars={
        "GOOGLE_GENAI_USE_VERTEXAI": "true",
        "GOOGLE_GEO_API_KEY": os.environ["GOOGLE_GEO_API_KEY"],
    },
)

print("Deployed agent resource name:", remote_agent.resource_name)

INFO:vertexai.agent_engines:Identified the following requirements: {'cloudpickle': '3.1.2', 'pydantic': '2.12.3', 'google-cloud-aiplatform': '1.157.0'}
INFO:vertexai.agent_engines:The following requirements are appended: {'pydantic==2.12.3', 'cloudpickle==3.1.2'}
INFO:vertexai.agent_engines:The final list of requirements: ['google-cloud-aiplatform[agent_engines,adk]', 'google-adk==1.29.0', 'pydantic==2.12.3', 'cloudpickle==3.1.2']
INFO:vertexai.agent_engines:Using bucket adroit-sol-501711-r0-adk-staging
INFO:vertexai.agent_engines:Wrote to gs://adroit-sol-501711-r0-adk-staging/agent_engine/agent_engine.pkl
INFO:vertexai.agent_engines:Writing to gs://adroit-sol-501711-r0-adk-staging/agent_engine/requirements.txt
INFO:vertexai.agent_engines:Creating in-memory tarfile of extra_packages
INFO:vertexai.agent_engines:Writing to gs://adroit-sol-501711-r0-adk-staging/agent_engine/dependencies.tar.gz
INFO:vertexai.agent_engines:Creating AgentEngine
INFO:vertexai.agent_engines:Create AgentEngine 

Deployed agent resource name: projects/735570248844/locations/us-central1/reasoningEngines/590638404987781120


## Testing the deployed agent

Same five capability paths, run against the live deployed service.

In [26]:
session = remote_agent.create_session(user_id="deploy_test_user")
print("Session created:", session["id"])

Session created: 8241565121697021952


In [27]:
# Test 1: Capability description
events = list(remote_agent.stream_query(
    user_id="deploy_test_user",
    session_id=session["id"],
    message="What can you do?",
))
print(f"Got {len(events)} events")
for event in events:
    print(event)

Got 1 events
{'model_version': 'gemini-2.5-flash', 'content': {'parts': [{'text': 'ReadyNow! is an emergency preparedness assistant. I can provide:\n\n*   **Real-time weather and alerts:** Get up-to-date weather conditions and emergency alerts for your location.\n*   **Current disaster news:** Stay informed about ongoing disasters and their impact.\n*   **Suggested evacuation routes:** Find safe routes in case of an evacuation.\n*   **General safety guidance:** Access information and tips for various emergency situations.', 'thought_signature': 'CrcCAY89a1_qzE5syhoFqV7SuPlOwvU_ld6MGyrGrMAXu-IzX9c3FnHrhHfyWqpcPdj534AuewGq1iItQnbJWVmTCLM1oQLwChdyisjlGLAyNH2bdpeh1ZurkHauZtQdoylrxH21pWovoFgxF6xaRdtubqg4JKqNmIoly8QBAu5OWYhiN5AiGcRSUmrCHkpy8aKOpLELqbWZdEDO_xCzWWnv2mqu5MAFPU8tQ_E6Mus3NKEGt5ayt2fdCgN8i0rLVHr_GdW7MPrV-lgo-BZnNT85Dll2mwOffzeC-qbE9LQHoMQcwtfLABy0A8m_QPUudpFF1cDpv5S1El9tduX4wc5zGYb15ugXv6162Lbytx250IFTxjYIaPYC4ygCt5myxN7KO7-A_0oei-n5A0ciCx0Os6or9km44jo='}], 'role': 'model'}, 'fini

In [28]:
# Test 2: Mission-scope refusal
events = list(remote_agent.stream_query(
    user_id="deploy_test_user",
    session_id=session["id"],
    message="Can you help me write a Python script to sort a list?",
))
print(f"Got {len(events)} events")
for event in events:
    print(event)

Got 1 events
{'model_version': 'gemini-2.5-flash', 'content': {'parts': [{'text': "I'm sorry, but I can only help with emergency preparedness and safety topics, such as weather alerts, disaster news, evacuation routes, and general safety guidance. I cannot assist with writing Python scripts or other coding-related tasks.", 'thought_signature': 'CtkCAY89a18rMWtoQDIGiak81kNJxoVdonbZr93V1iSdnvGGm5lM1t11fQ6HNlffIiZpCUHq_-A6yKd9Hu2NYtV5Lt4cZA8d_wtMao14PI33WrosuQLMpCK8bIMgFno_cP9HPzEqIbGAhYh7PjtTX_UBavzBIgTiiyjqVy2omeDWNqWGByDudL_KMWpn014t3FojoBKRiNfAk0Sf0A3gGLNdXM90NoR6nZdaQX5RLVAk3cqn5Pez4zLfeGxcCLvEdtdgwyUhjYf_4e_U6fZRRnwOcx5yKelsb-GzKSQH2MkIVUyfUTBW-rf0hjF5LAwfLH-6x3qc56SUnz7KCrHkOP95srDOkJxEOTYeB0JMm8Q5lcI8ARgjys4dGGLobB62aGmjp6wTEh2elEu-6BYSZcO9ma6N62bNovTtnPEGaZJPcKivRoMpIvJEEFu2f31wm_89JsUzfN0zVbyn'}], 'role': 'model'}, 'finish_reason': 'STOP', 'usage_metadata': {'candidates_token_count': 47, 'candidates_tokens_details': [{'modality': 'TEXT', 'token_count': 47}], 'prompt_token_count'

In [29]:
# Test 3: Weather/alerts
events = list(remote_agent.stream_query(
    user_id="deploy_test_user",
    session_id=session["id"],
    message="What's the weather like in Kansas City, MO right now, any alerts?",
))
print(f"Got {len(events)} events")
for event in events:
    print(event)

Got 7 events
{'model_version': 'gemini-2.5-flash', 'content': {'parts': [{'function_call': {'id': 'adk-473fd87c-9460-4bdd-bcd3-17d0ddc2ed76', 'args': {'agent_name': 'weather_team'}, 'name': 'transfer_to_agent'}, 'thought_signature': 'CrACAY89a1_mS6HrCbX9Zer2S-3A7N9qBD4P3DQOHZ6m6AMq2kkd1kFmDHgJV_iHu1g8i36IXN9KYqdw4MeeWyAgjb6I2spDKccxN3N82WJ0GkYQDlctQR8JnvghymIqGJIAmftHX1dUjKVEuTzMEXFaPiT42RJoE09XttJeFaq-8PkGEBtuB1KlLMh_jqwtRZMephZanzKH3dS0CCKE3DB9ZNioEXVi5ezlnfXB6kRZaF3megRetqOwfJuD8eQlYFUbHqUTAb9xl9SNaUd64lCYpeqUOj9wawh3DBu9MM17mqSCZdp76XTVLXlMC-29LvRfukXLhTkzrh3ubEjWHwaz3ld0A0qRw16bIk1GU0hTlLCNePoL10kbXsM09CfoZl1875PzdUrefD3BwpM3Sg-EJw=='}], 'role': 'model'}, 'finish_reason': 'STOP', 'usage_metadata': {'candidates_token_count': 11, 'candidates_tokens_details': [{'modality': 'TEXT', 'token_count': 11}], 'prompt_token_count': 859, 'prompt_tokens_details': [{'modality': 'TEXT', 'token_count': 859}], 'thoughts_token_count': 57, 'total_token_count': 927, 'traffic_type': 'ON_DEMAND'}, 'avg_

In [30]:
# Test 4: Routes
events = list(remote_agent.stream_query(
    user_id="deploy_test_user",
    session_id=session["id"],
    message="I need a safe route from Kansas City, MO to Topeka, KS",
))
print(f"Got {len(events)} events")
for event in events:
    print(event)

Got 7 events
{'model_version': 'gemini-2.5-flash', 'content': {'parts': [{'function_call': {'id': 'adk-239c340f-7d74-4ef9-9f8e-697e022cad45', 'args': {'agent_name': 'routes_team'}, 'name': 'transfer_to_agent'}, 'thought_signature': 'CqgCAY89a19KxKhsgZLX4D096TvBGsgouW9nrmOX7sLECjigFJiX38DJNz3JcczuIExvo2lGUEswnqHnqLkvQxGSDnsgtl2uMCQUHqclix_t7_XOwzfyHOBvfCPe_asIDNHWbZCKtkR17k1-YP1Ua-XwZmdgAwi1w4VchZ6N10F3oHVOwb8t6wdhzKRd7bm9uCiZqkWme6Fal-KP6JNzgmJsCPXcIWPslnhiE2GiB0P5Ox7kV05rHDgKEAfweeOiXhnNdWY3E8nrK5Q0XjzvTWQJvpxyjJ_uStunKle_5N84eI7gpGwuEfuRxnRstbjdL1PALXnpXcleUJdJsNCd_Lw6K5HAZMHODMJ_SYI-m1PW0587S2K9sIJ4OfV-yiCxBeXIeOLkW9YiS2A='}], 'role': 'model'}, 'finish_reason': 'STOP', 'usage_metadata': {'candidates_token_count': 11, 'candidates_tokens_details': [{'modality': 'TEXT', 'token_count': 11}], 'prompt_token_count': 1365, 'prompt_tokens_details': [{'modality': 'TEXT', 'token_count': 1365}], 'thoughts_token_count': 56, 'total_token_count': 1432, 'traffic_type': 'ON_DEMAND'}, 'avg_logprobs':

In [31]:
# Test 5: Search/news
events = list(remote_agent.stream_query(
    user_id="deploy_test_user",
    session_id=session["id"],
    message="Are there any major wildfires happening right now?",
))
print(f"Got {len(events)} events")
for event in events:
    print(event)

Got 5 events
{'model_version': 'gemini-2.5-flash', 'content': {'parts': [{'function_call': {'id': 'adk-8b67fc15-cadb-4b66-bf4c-f11f0a812b73', 'args': {'agent_name': 'search_team'}, 'name': 'transfer_to_agent'}, 'thought_signature': 'CuACAY89a1_i-WuxXfedKJdKlsb5XDBFkjIfPd0kB37e8LjFPpYi06Oac_0NFmcQi68zIm_SITAOOXO2Ettaz2mcAA2jUztt_Nocvfvu1I0ajhRHP_RXM_rgt7E-SrqFZYxmyvh6N68EYp-fseZdkJwWtiBMX-2DEWx-_f1nmza5zLoBoKWg4JWqGDwqv10KsyTFmnCEJIrXu29hodyyYNG2PCjK5gN_LIL23ODfjrDwwmWk5-DgbUaUMJkUoOjJdcHonCC7WX2a14Gi6wYR_Lorr7ISEHNCnH4Yxy9eqeAChYvaH0qpr0Lq1--gmzOglaikzDlLuGgrAlUVp7bhb5JRcrBg0fXNJdu_p94XS9g812dxCQRdDwf9HjpG_uHl7ARcj4dDXc1owi7bokM4G9K9VqegAKxC1kfTozTKRJHx3KwuRQcLjhnKSYkGl0wHl6BWxHitE57VggmoBxVdoJqscg=='}], 'role': 'model'}, 'finish_reason': 'STOP', 'usage_metadata': {'candidates_token_count': 11, 'candidates_tokens_details': [{'modality': 'TEXT', 'token_count': 11}], 'prompt_token_count': 2889, 'prompt_tokens_details': [{'modality': 'TEXT', 'token_count': 2889}], 'thoughts_token_count': 